In [44]:
import sys
import os
import urllib3

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)

Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [45]:
import pandas as pd
from datetime import datetime, timedelta
import pytz

# Define time period
# Calculate the start date (2 days ago) at 00:00:00 UTC
start_date = (datetime.now(pytz.UTC) - timedelta(days=2)).date()

# Format as 'YYYY-MM-DDT00:00:00Z'
start = f"{start_date}T00:00:00Z"

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        observed_src = observed_src[observed_src['lastObserved'] >= start]
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")

display(observed_src)


Querying owner: HTOC Org

Retrieved 808 unique observed indicators.


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,associatedGroups.data,hostName,dnsActive,whoisActive,text,address,url,sha256,Block,indicator
0,5629499566213979,2025-09-04T14:20:28Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-20T15:41:59Z,3.0,69.0,INC9223440,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.159.71.249
1,6755399481093384,2025-11-19T12:50:50Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-20T15:31:59Z,3.0,80.0,RITM0626208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,79.127.181.22
2,5629499563360156,2025-08-13T15:08:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-20T15:31:59Z,3.0,66.0,RITM0606742,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,176.65.148.184
4,6755399459033757,2025-06-16T17:42:40Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-20T15:07:38Z,3.0,68.0,TASK0912447,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,179.43.159.200
22,5629499555060668,2025-06-16T17:42:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-20T15:07:38Z,3.0,58.0,RITM0589984,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,179.43.159.201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9819,5629499556005878,2025-06-30T12:21:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-17T08:36:59Z,3.0,60.0,RITM0594014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.114
9820,5629499556005877,2025-06-30T12:21:42Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-17T08:36:59Z,3.0,60.0,RITM0594014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.182
9821,5629499556005870,2025-06-30T12:21:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-17T08:36:59Z,3.0,60.0,RITM0594014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.62.156.49
9822,5629499556005850,2025-06-30T12:21:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-17T08:36:59Z,3.0,60.0,RITM0594014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.62.156.47


In [46]:
observed_src[observed_src['indicator'] == '208.115.228.234']

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,associatedGroups.data,hostName,dnsActive,whoisActive,text,address,url,sha256,Block,indicator
799,5629499574089565,2025-10-21T11:33:56Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-20T13:56:59Z,5.0,96.0,Key Findings\nSilent Push Threat Analysts have...,...,"[{'id': 6755399470002411, 'dateAdded': '2025-1...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,208.115.228.234


In [47]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251120.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251119.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251118.csv']


C:\Users\jaskew\AppData\Local\Temp\ipykernel_8692\564055639.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Loaded data from 3 files.


In [48]:
observed_data_df[observed_data_df['indicator'] == '208.115.228.234']

,indicator,API_UserName,obs_date,OpDiv,indicator_key,observations,curr_date
5834,208.115.228.234,50189120947314147395,2025-11-19,NIH,208.115.228.234_NIH,8,2025-11-19
5835,208.115.228.234,74198686107399967946,2025-11-19,VA,208.115.228.234_VA,1,2025-11-19


In [69]:
import pandas as pd
from datetime import timedelta
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION & SETUP
# ═══════════════════════════════════════════════════════════════════════════════

# Time cutoffs
cutoff = pd.Timestamp.utcnow()
cutoff_naive = cutoff.tz_convert(None)

# Required columns validation
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [c for c in required_columns if c not in observed_data_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def process_tag_row(row, observed_src):
    """Process a single row from observed_src to extract and filter tags."""
    tags_data = row.get('tags.data')
    if not isinstance(tags_data, list):
        return None

    # Normalize the tags for the current row
    tags_df = pd.json_normalize(tags_data)

    # Ensure string and apply VA rename before filtering
    tags_df['name'] = (
        tags_df['name']
        .astype(str)
        .str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)
    )

    # Skip if all associated groups are Adversary
    if 'associatedGroups.data' in observed_src.columns:
        ag_data = row.get('associatedGroups.data')
        if isinstance(ag_data, list) and len(ag_data) > 0:
            groups_df = pd.json_normalize(ag_data)
            if 'type' in groups_df.columns and set(groups_df['type']) == {'Adversary'}:
                return None

    # All tag names (for all_tags)
    all_tags_list = tags_df['name'].tolist()

    # Filter for API tags
    api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)].copy()
    if api_tags.empty:
        return None

    # Add metadata columns
    meta_cols = [
        'summary', 'observations', 'description', 'type',
        'dateAdded', 'lastModified', 'lastObserved', 'webLink',
        'rating', 'confidence', 'id', 'associatedGroups.data'
    ]
    for col in meta_cols:
        api_tags[col] = [row.get(col)] * len(api_tags)

    # Attach all tags list
    if len(api_tags) > 0:
        api_tags['all_tags'] = [all_tags_list] * len(api_tags)

    return api_tags

def map_observed_dates(filtered_tags, observed_data_df):
    """Map observed dates from observed_data_df to filtered_tags."""
    if filtered_tags.empty:
        return filtered_tags
    
    # Clean name -> match OpDiv (remove trailing ' Splunk API')
    filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)
    filtered_tags['observed_date'] = pd.NaT
    
    # Ensure observed_data_df obs_date is datetime (naive)
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')

    for idx, r in filtered_tags.iterrows():
        summary = r.get('summary')
        cleaned_name = r.get('cleaned_name')
        if pd.isna(summary) or pd.isna(cleaned_name):
            continue
        match = observed_data_df[
            (observed_data_df['indicator'] == summary) &
            (observed_data_df['OpDiv'] == cleaned_name)
        ]
        if not match.empty:
            filtered_tags.at[idx, 'observed_date'] = match['obs_date'].iloc[0]

    filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')
    
    # Drop helper column
    filtered_tags.drop(columns=['cleaned_name'], inplace=True, errors='ignore')
    
    return filtered_tags

def get_multi_partner_indicators(observed_data_df, cutoff_naive):
    """Get indicators that have multiple partners from observed_data_df."""
    if observed_data_df.empty:
        return pd.DataFrame()
    
    # Ensure obs_date is datetime
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')
    
    # Filter by recent dates (last 3 days)
    recent_obs = observed_data_df[
        observed_data_df['obs_date'] >= cutoff_naive - timedelta(days=3)
    ].copy()
    
    if recent_obs.empty:
        return pd.DataFrame()
    
    # Group by indicator and count unique OpDiv (partners)
    partner_counts = (
        recent_obs.groupby('indicator')['OpDiv']
        .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
        .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
    )
    
    # Keep only indicators with multiple partners
    multi_partner_indicators = partner_counts[partner_counts['partner_count'] >= 2].copy()
    
    return multi_partner_indicators

def apply_filters_and_grouping(filtered_tags, cutoff_naive, multi_partner_indicators):
    """Apply time filters, partner grouping, and other filtering criteria."""
    if filtered_tags.empty:
        return pd.DataFrame()
    
    # Filter by recent observed_date (keep date filtering)
    recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()

    if recent_tags.empty:
        return recent_tags
    
    # Filter to only include indicators that have multiple partners in observed_data_df
    if not multi_partner_indicators.empty:
        recent_tags = recent_tags[
            recent_tags['summary'].isin(multi_partner_indicators['indicator'])
        ].copy()
    
    if recent_tags.empty:
        return recent_tags
    
    # Partner extraction and grouping from ThreatConnect tags (as fallback)
    recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

    partner_groups = (
        recent_tags.groupby('summary')['partner']
        .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
        .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
    )

    recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')
    
    # Merge with multi_partner_indicators to get partners from observed_data_df
    if not multi_partner_indicators.empty:
        recent_tags = recent_tags.merge(
            multi_partner_indicators[['indicator', 'partners', 'partner_count']],
            left_on='summary',
            right_on='indicator',
            how='left',
            suffixes=('', '_from_obs')
        )
        # Use partners from observed_data_df if available, otherwise use from tags
        recent_tags['partners'] = recent_tags['partners_from_obs'].fillna(recent_tags['partners'])
        recent_tags['partner_count'] = recent_tags['partner_count_from_obs'].fillna(recent_tags['partner_count'])
        recent_tags = recent_tags.drop(columns=['indicator', 'partners_from_obs', 'partner_count_from_obs'], errors='ignore')

    # Additional filters - keep only indicators with multiple partners
    recent_tags = recent_tags[recent_tags['partner_count'] >= 2]

    # Deduplicate by summary
    recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

    # Drop unneeded columns if present
    cols_to_drop = [
        'techniqueId', 'tactics.data', 'tactics.count',
        'platforms.data', 'platforms.count', 'partner', 'name'
    ]
    recent_tags = recent_tags.drop(columns=[c for c in cols_to_drop if c in recent_tags.columns], errors='ignore')

    # Remove rows where all_tags contains unwanted markers (keep htoc_wl filter, but not I&W)
    if 'all_tags' in recent_tags.columns:
        recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'htoc_wl' in x)]
    
    # Add partners from tags at the end (after all filtering)
    # Re-extract partners from tags for the final filtered dataset
    if not recent_tags.empty and 'all_tags' in recent_tags.columns:
        def extract_partners_from_tags(all_tags_list):
            """Extract partner names from tags that contain 'API'."""
            if not isinstance(all_tags_list, list):
                return []
            api_partners = []
            for tag in all_tags_list:
                if isinstance(tag, str) and 'API' in tag:
                    # Extract partner name (remove ' Splunk API' suffix)
                    partner = tag.replace(' Splunk API', '').replace('VA CSOC CTS Splunk', 'VA').strip()
                    if partner:
                        api_partners.append(partner)
            return sorted(set(api_partners))
        
        # Extract partners from tags for each indicator
        tag_partners_series = recent_tags['all_tags'].apply(extract_partners_from_tags)
        
        # Combine with existing partners from observed_data_df
        def combine_partners(row_idx):
            """Combine partners from observed_data_df and tags for a specific row."""
            obs_partners = recent_tags.loc[row_idx, 'partners']
            tag_partners_list = tag_partners_series.loc[row_idx]
            
            combined = set()
            # Add partners from observed_data_df
            if pd.notna(obs_partners) and obs_partners:
                for p in str(obs_partners).split(', '):
                    if p.strip():
                        combined.add(p.strip())
            # Add partners from tags
            if tag_partners_list:
                for p in tag_partners_list:
                    if p:
                        combined.add(p)
            return ', '.join(sorted(combined)) if combined else obs_partners
        
        recent_tags['partners'] = recent_tags.index.to_series().apply(combine_partners)
        
        # Update partner_count based on combined partners
        recent_tags['partner_count'] = recent_tags['partners'].apply(
            lambda x: len([p for p in str(x).split(', ') if p.strip()]) if pd.notna(x) and x else 0
        )

    return recent_tags

def extract_group_ids(recent_tags):
    """Extract group IDs from associatedGroups.data."""
    if 'associatedGroups.data' in recent_tags.columns:
        recent_tags['group_id'] = recent_tags['associatedGroups.data'].apply(
            lambda x: x[0]['id'] if isinstance(x, list) and len(x) > 0 and 'id' in x[0] else None
        )
        # Convert group_id to string type and remove trailing decimals if it exists
        if 'group_id' in recent_tags.columns:
            recent_tags['group_id'] = recent_tags['group_id'].apply(
                lambda x: str(int(float(x))) if pd.notna(x) and str(x) != 'None' else x
            ).astype(str)
    return recent_tags

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN PROCESSING PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

print("Starting tag processing pipeline...")

# Step 1: Process tags from observed_src
print("Processing tags from observed_src...")
all_filtered = []

for _, row in observed_src.iterrows():
    processed_tags = process_tag_row(row, observed_src)
    if processed_tags is not None:
        all_filtered.append(processed_tags)

# Step 2: Create filtered_tags DataFrame
print("Creating filtered_tags DataFrame...")
filtered_tags = pd.concat(all_filtered, ignore_index=True) if all_filtered else pd.DataFrame()

if not filtered_tags.empty:
    # Ensure datetime columns
    filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce', utc=True)
    filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce', utc=True)
    print(f"Created filtered_tags with {len(filtered_tags)} rows")
else:
    print("No filtered tags found")

# Step 3: Map observed dates
print("Mapping observed dates...")
filtered_tags = map_observed_dates(filtered_tags, observed_data_df)

# Step 3.5: Get indicators with multiple partners from observed_data_df
print("Identifying indicators with multiple partners from observed_data_df...")
multi_partner_indicators = get_multi_partner_indicators(observed_data_df, cutoff_naive)
print(f"Found {len(multi_partner_indicators)} indicators with multiple partners")

# Step 4: Apply filters and grouping
print("Applying filters and partner grouping...")
recent_tags = apply_filters_and_grouping(filtered_tags, cutoff_naive, multi_partner_indicators)

# Step 5: Extract group IDs
print("Extracting group IDs...")
recent_tags = extract_group_ids(recent_tags)
# Step 6: Add I&W column indicating if indicator has been tagged as 'I&W' (used in previous report)
if 'all_tags' in recent_tags.columns:
    recent_tags['I&W'] = recent_tags['all_tags'].apply(
        lambda x: 'Yes' if isinstance(x, list) and 'I&W' in x else 'No'
    )
else:
    recent_tags['I&W'] = 'No'

# Move I&W column to the very end
cols = [col for col in recent_tags.columns if col != 'I&W'] + ['I&W']
recent_tags = recent_tags[cols]

# Final summary
print(f"Processing complete! Final dataset shape: {recent_tags.shape}")
if not recent_tags.empty:
    print(f"Partners represented: {recent_tags['partners'].nunique()} unique partner combinations")

display(recent_tags)

Starting tag processing pipeline...
Processing tags from observed_src...
Creating filtered_tags DataFrame...
Created filtered_tags with 5783 rows
Mapping observed dates...
Identifying indicators with multiple partners from observed_data_df...
Found 699 indicators with multiple partners
Applying filters and partner grouping...
Extracting group IDs...
Processing complete! Final dataset shape: (513, 19)
Partners represented: 51 unique partner combinations


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id,I&W
0,5629499566213979,2025-11-12T16:27:29Z,INC9223440,121.159.71.249,49304,Address,2025-09-04 14:20:28+00:00,2025-11-20T15:41:59Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,69.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-20,8,"CMS, DHA, FDA, HHS, IHS, NIH, OS, VA",nan,No
2,5629499546480613,2025-11-12T16:27:29Z,TASK0882701 / RITM0585661,78.153.140.224,149703,Address,2025-06-02 18:33:32+00:00,2025-11-20T15:07:37Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,56.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-20,8,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS",nan,No
4,5629499541090468,2025-11-13T07:08:13Z,FBI Email Alert May 14 2025 Medium IP IOCs,104.152.52.139,1771,Address,2025-05-14 17:47:33+00:00,2025-11-20T15:07:37Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,55.0,NaN,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-11-20,6,"CMS, FDA, HHS, HRSA, OS, VA",nan,No
5,4697035,2025-11-12T00:25:06Z,"Starting in April 2024, UNC5537 used stolen cr...",154.47.30.137,1748,Address,2024-06-05 13:07:49+00:00,2025-11-20T15:07:36Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,72.0,"[{'id': 6755399470002842, 'dateAdded': '2025-1...","[DHA Splunk API, Snowflake, UNC5537, Credentia...",2025-11-19,8,"CMS, DHA, FDA, HHS, HRSA, NIH, OS, VA",6755399470002842,Yes
6,4697065,2025-11-12T16:27:29Z,"Starting in April 2024, UNC5537 used stolen cr...",37.19.210.21,7108,Address,2024-06-05 13:07:49+00:00,2025-11-20T15:07:35Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,71.0,"[{'id': 6755399470002842, 'dateAdded': '2025-1...","[DHA Splunk API, Snowflake, UNC5537, Credentia...",2025-11-20,9,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",6755399470002842,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3456,5629499556005878,2025-11-13T07:08:20Z,RITM0594014,65.49.1.114,2147920,Address,2025-06-30 12:21:43+00:00,2025-11-17T08:36:59Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,60.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-20,10,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",nan,No
3465,5629499556005877,2025-11-13T07:08:20Z,RITM0594014,65.49.1.182,1366051,Address,2025-06-30 12:21:42+00:00,2025-11-17T08:36:59Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,60.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-20,9,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",nan,No
3474,5629499556005870,2025-11-13T07:08:20Z,RITM0594014,64.62.156.49,2172433,Address,2025-06-30 12:21:37+00:00,2025-11-17T08:36:59Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,60.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-20,10,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",nan,No
3482,5629499556005850,2025-11-13T07:08:20Z,RITM0594014,64.62.156.47,2055126,Address,2025-06-30 12:21:18+00:00,2025-11-17T08:36:59Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,60.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-20,10,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",nan,No


In [70]:
recent_tags[recent_tags['summary'] == '23.106.161.1']

,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id,I&W
234,5629499574089567,2025-11-13T07:08:13Z,Key Findings\nSilent Push Threat Analysts have...,23.106.161.1,17,Address,2025-10-21 11:33:56+00:00,2025-11-20T13:56:41Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,96.0,"[{'id': 6755399470002411, 'dateAdded': '2025-1...","[OtterCookie, Contagious Interview, Illicit IT...",2025-11-19,3,"DHA, NIH, VA",6755399470002411,Yes


In [51]:
import pandas as pd

# Extract unique indicators from recent_tags
indicators = recent_tags['summary'].unique()

# Initialize a list to store attribute data
attributes_data = []

# Track indicators with no attributes
indicators_with_no_attributes = []

# Iterate over unique indicators
for indicator in indicators:
    try:
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            data = response.json().get('data', {})
            attributes = data.get('attributes', {}).get('data', [])

            if not attributes:
                print(f"No attributes found for indicator: {indicator}")
                # Track indicators with no attributes
                indicators_with_no_attributes.append(indicator)
            else:
                for attr in attributes:
                    attr.update({
                        'id': data.get('id'),
                        'summary': data.get('summary'),
                        'type': data.get('type'),
                        'ownerName': data.get('ownerName')
                    })
                    attributes_data.append(attr)
        # Suppress non-JSON and known 400 error responses
    except Exception as e:
        # Suppress error output, including known 400 error
        if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
            continue
        if "Status Code: 400" in str(e):
            continue
        pass

# Create attributes 
attributes_observed_src = pd.json_normalize(attributes_data)

# Un-nest 'createdBy' and filter out 'SOAR' entries
if not attributes_observed_src.empty and attributes_observed_src['createdBy.lastName'].notnull().any():
    attributes_observed_src = attributes_observed_src[attributes_observed_src['createdBy.lastName'] != 'SOAR']

# Drop duplicates based on 'id' to keep unique attribute records
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

# Filter recent_tags for indicators that had attributes
filtered_with_attrs = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])]

# Filter recent_tags for indicators that had no attributes
no_attrs_df = recent_tags[recent_tags['summary'].isin(indicators_with_no_attributes)]

# Combine both and remove duplicates based on 'summary'
filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)
display(filtered_recent_tags)


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"More than one indicator matches the criteria you specified. Try looking up by ID instead.","status":"Error"}'


No attributes found for indicator: ctrk.klclick2.com


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"More than one indicator matches the criteria you specified. Try looking up by ID instead.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"More than one indicator matches the criteria you specified. Try looking up by ID instead.","status":"Error"}'


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id
0,4697035,2025-11-12T00:25:06Z,"Starting in April 2024, UNC5537 used stolen cr...",154.47.30.137,1748,Address,2024-06-05 13:07:49+00:00,2025-11-20T15:07:36Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,72.0,"[{'id': 6755399470002842, 'dateAdded': '2025-1...","[DHA Splunk API, Snowflake, UNC5537, Credentia...",2025-11-19,3,"CMS, FDA, VA",6755399470002842
1,4697065,2025-11-12T16:27:29Z,"Starting in April 2024, UNC5537 used stolen cr...",37.19.210.21,7108,Address,2024-06-05 13:07:49+00:00,2025-11-20T15:07:35Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,71.0,"[{'id': 6755399470002842, 'dateAdded': '2025-1...","[DHA Splunk API, Snowflake, UNC5537, Credentia...",2025-11-20,5,"CMS, DHA, FDA, HRSA, OS",6755399470002842
2,4478644,2025-11-13T07:08:13Z,ACD R&F processed a malspam campaign with an I...,kerry@govwhitepapers.com,223,EmailAddress,2023-11-21 18:07:18+00:00,2025-11-20T14:06:59Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,2.0,45.0,"[{'id': 288608, 'dateAdded': '2023-11-21T22:02...","[hhs-2023112102, govwhitepapers, Phishing, VA ...",2025-11-19,2,"HHS, VA",288608
3,5629499574089565,2025-11-13T07:08:13Z,Key Findings\nSilent Push Threat Analysts have...,208.115.228.234,32,Address,2025-10-21 11:33:56+00:00,2025-11-20T13:56:59Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,96.0,"[{'id': 6755399470002411, 'dateAdded': '2025-1...","[OtterCookie, Contagious Interview, Illicit IT...",2025-11-19,2,"NIH, VA",6755399470002411
4,5629499574089567,2025-11-13T07:08:13Z,Key Findings\nSilent Push Threat Analysts have...,23.106.161.1,17,Address,2025-10-21 11:33:56+00:00,2025-11-20T13:56:41Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,96.0,"[{'id': 6755399470002411, 'dateAdded': '2025-1...","[OtterCookie, Contagious Interview, Illicit IT...",2025-11-19,3,"DHA, NIH, VA",6755399470002411
5,6755399443015052,2025-11-13T05:43:04Z,CMS received a Volumetrics alert regarding mul...,91.196.152.67,8928307,Address,2025-03-14 11:55:20+00:00,2025-11-20T09:37:00Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,44.0,"[{'id': 6755399443002242, 'dateAdded': '2025-0...","[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-20,2,"CMS, HRSA",6755399443002242
6,6755399443015048,2025-11-13T05:43:04Z,CMS received a Volumetrics alert regarding mul...,91.196.152.66,8951178,Address,2025-03-14 11:55:19+00:00,2025-11-20T09:37:00Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,44.0,"[{'id': 6755399443002242, 'dateAdded': '2025-0...","[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-20,2,"CMS, HRSA",6755399443002242
7,6755399443015047,2025-11-13T05:43:04Z,CMS received a Volumetrics alert regarding mul...,91.196.152.46,9212679,Address,2025-03-14 11:55:19+00:00,2025-11-20T09:37:00Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,45.0,"[{'id': 6755399443002242, 'dateAdded': '2025-0...","[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-20,2,"CMS, HRSA",6755399443002242
8,6755399443015046,2025-11-13T05:43:04Z,CMS received a Volumetrics alert regarding mul...,91.196.152.71,8961058,Address,2025-03-14 11:55:19+00:00,2025-11-20T09:37:00Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,44.0,"[{'id': 6755399443002242, 'dateAdded': '2025-0...","[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-20,2,"CMS, HRSA",6755399443002242
9,6755399443015044,2025-11-13T05:43:04Z,CMS received a Volumetrics alert regarding mul...,91.196.152.45,9041427,Address,2025-03-14 11:55:19+00:00,2025-11-20T09:37:00Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,44

In [65]:
import os, io, csv
import pandas as pd

# ── Path to the reported indicators file ──────────────────────────────────────
reported_indicators_path = r"Z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Reported Indicators\indicators.csv"

# ── Robust loader: returns (list_of_nonempty_values, column_used) ─────────────
def _load_indicator_values(path: str, col: str = "Indicator"):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    # 1) Try pandas with delimiter sniffing (no noisy warnings)
    try:
        df = pd.read_csv(
            path,
            sep=None,
            engine="python",
            encoding="utf-8-sig",
            dtype=str
        )
        if col not in df.columns:
            # Case-insensitive fallback
            ci_match = [c for c in df.columns if c.strip().lower() == col.lower()]
            if ci_match:
                col = ci_match[0]
            else:
                raise KeyError(f"Column '{col}' not found. Columns: {list(df.columns)}")

        ser = (
            df[col]
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .dropna()
        )
        return ser.tolist(), col

    except Exception:
        # 2) Fallback to csv.Sniffer for very messy files
        with open(path, "rb") as fh:
            raw = fh.read()

        text = None
        for enc in ("utf-8-sig", "utf-8", "latin-1"):
            try:
                text = raw.decode(enc)
                break
            except UnicodeDecodeError:
                continue
        if text is None:
            raise UnicodeDecodeError("utf-8/latin-1", b"", 0, 1, "Could not decode file")

        sample = text[:4096]
        try:
            dialect = csv.Sniffer().sniff(sample)
            has_header = csv.Sniffer().has_header(sample)
        except Exception:
            dialect = csv.excel
            dialect.delimiter = ","
            dialect.skipinitialspace = True
            has_header = True

        if has_header:
            reader = csv.DictReader(io.StringIO(text), dialect=dialect)
            header = [h.strip().lstrip("\ufeff") for h in reader.fieldnames or []]
            # normalize to located column name actually present
            if col not in header:
                ci_match = [h for h in header if h.lower() == col.lower()]
                if ci_match:
                    col = ci_match[0]
                else:
                    raise KeyError(f"Column '{col}' not found. Columns: {header}")

            vals = []
            for row in reader:
                if not row:
                    continue
                v = (row.get(col) or "").strip()
                if v:
                    vals.append(v)
            return vals, col
        else:
            # no header, assume first column is Indicator
            rdr = csv.reader(io.StringIO(text), dialect=dialect)
            vals = []
            for parts in rdr:
                if parts:
                    v = (parts[0] or "").strip()
                    if v:
                        vals.append(v)
            return vals, col  # col label remains as requested

# ── Load reported indicators and filter out already-reported indicators ───────
try:
    values, col_name = _load_indicator_values(reported_indicators_path, col="Indicator")
    print(f"Loaded reported indicators — non-empty rows in '{col_name}': {len(values)}")

    # Unique set for filtering
    reported_set = set(values)

    # Filter out already-reported indicators from filtered_recent_tags
    if 'filtered_recent_tags' in globals() and not filtered_recent_tags.empty and reported_set:
        before_count = len(filtered_recent_tags)
        filtered_recent_tags = filtered_recent_tags[
            ~filtered_recent_tags['summary'].astype(str).isin(reported_set)
        ].reset_index(drop=True)
        after_count = len(filtered_recent_tags)
        print(f"Removed {before_count - after_count} already-reported indicators")
        print(f"Final filtered dataset: {after_count} indicators")
    elif 'filtered_recent_tags' not in globals():
        print("Warning: filtered_recent_tags not found - run previous cells first")
    else:
        print("No filtering applied (empty dataset or no reported indicators).")

except Exception as e:
    print(f"[ERROR] {e}")

# ── Display final filtered results ────────────────────────────────────────────
if 'filtered_recent_tags' in globals():
    display(filtered_recent_tags)
else:
    print("No filtered_recent_tags to display")


Loaded reported indicators — non-empty rows in 'Indicator': 249
Removed 1 already-reported indicators
Final filtered dataset: 31 indicators


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id
0,6755399459033712,The IP is hosted by Linode (Akamai Connected C...,2025-10-09T16:20:09Z,172.233.190.104,98719141,Address,2025-06-16 17:35:08+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,2.0,78.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,5,"CMS, DHA, HHS, HRSA, NIH",nan
1,6755399459033709,"IP 45.33.69.253 is a high-confidence, high-ris...",2025-10-09T16:23:15Z,45.33.69.253,102464235,Address,2025-06-16 17:35:07+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,38.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
2,6755399459033706,IP address 172.233.195.143 has been flagged fo...,2025-10-09T16:23:15Z,172.233.195.143,90817880,Address,2025-06-16 17:35:05+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,48.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
3,6755399459033703,IP address 172.232.188.177 has been identified...,2025-10-09T16:23:15Z,172.232.188.177,101520385,Address,2025-06-16 17:35:02+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,41.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
4,6755399459033700,172.234.207.202 is a critically rated maliciou...,2025-10-09T16:23:15Z,172.234.207.202,104833938,Address,2025-06-16 17:35:01+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,68.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
5,5629499555060654,IP address 172.234.246.109 has been flagged fo...,2025-10-09T16:23:15Z,172.234.246.109,91902099,Address,2025-06-16 17:35:07+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,2.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
6,5629499555060653,66.175.208.166 is a high-risk IP currently hos...,2025-10-09T16:20:09Z,66.175.208.166,105435530,Address,2025-06-16 17:35:06+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,73.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,5,"CMS, DHA, HHS, HRSA, NIH",nan
7,5629499554313271,IP address 104.237.151.205 has been heavily ob...,2025-10-09T16:20:09Z,104.237.151.205,107059729,Address,2025-06-11 14:47:15+00:00,2025-10-09T17:13:19Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,68.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,5,"CMS, DHA, HHS, HRSA, NIH",nan
8,4809962,Sharing malicious indicators flagged by Virust...,2025-10-09T16:23:15Z,165.22.54.16,1947152,Address,2024-08-08 18:17:34+00:00,2025-10-09T16:36:24Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,24.0,"[{'id': 447870, 'dateAdded': '2024-08-08T18:17...","[Indicators, Malicious Activity, OS Splunk API...",2025-10-09,2,"CMS, HHS",447870
9,6755399460008411,"Details\nIn late June 2025, Iran-aligned hackt...",2025-10-09T16:20:09Z,200.31.12.168,159,Address,2025-07-02 12:05:37+00:00,2025-10-09T14:14:16Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,2.0,60.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-10-08,2,"DHA, VA",5629499544001057


In [66]:
from datetime import datetime, UTC

output_path = 'Z:\\HTOC\\HTOC Reports\\I&W Reports\\5. I&W Staging\\Spreadsheet\\Expanded'

# Generate filename with today's date in YYYYMMDD format
today_str = datetime.now(UTC).strftime("%Y%m%d")
excel_filename = f"expanded_indicators_{today_str}.xlsx"
excel_path = os.path.join(output_path, excel_filename)

# Convert all timezone-aware datetime columns to naive (remove timezone info)
for col in filtered_recent_tags.select_dtypes(include=['datetimetz']).columns:
    filtered_recent_tags[col] = filtered_recent_tags[col].dt.tz_localize(None)

# Drop the 'description' column if it exists
if 'description' in filtered_recent_tags.columns:
    filtered_recent_tags = filtered_recent_tags.drop(columns=['description'])

filtered_recent_tags.to_excel(excel_path, index=False)
print(f"Saved to {excel_path}")

Saved to Z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Spreadsheet\Expanded\expanded_indicators_20251009.xlsx
